# 02 — Leakage-Resistant Split Protocol

Mục tiêu: tạo một protocol đánh giá ổn định cho 5,132 ảnh đã audit.

- 15% dữ liệu được khóa làm **internal holdout** và không dùng để tuning.
- 85% còn lại được chia thành **5 folds** để cross-validation.
- Các ảnh exact duplicate (cùng SHA-256) luôn ở cùng split/fold để tránh leakage.
- Notebook chỉ tạo manifest CSV/JSON; **không di chuyển hoặc sửa ảnh gốc**.


In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd

SEED = 2026
HOLDOUT_FRACTION = 0.15
N_FOLDS = 5
PROJECT_ROOT = Path(r'D:\medicine')
AUDIT_CSV = PROJECT_ROOT / 'audit_outputs' / 'image_audit.csv'
OUTPUT_DIR = PROJECT_ROOT / 'split_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
print('Audit CSV:', AUDIT_CSV, '| exists=', AUDIT_CSV.exists())
print('Output dir:', OUTPUT_DIR)


## 1. Load audited dataset and enforce integrity gates

In [ ]:
df = pd.read_csv(AUDIT_CSV)
required = {'label','filename','path','read_ok','sha256'}
missing = required - set(df.columns)
if missing: raise ValueError(f'Missing required columns: {missing}')
if len(df) != 5132: raise ValueError(f'Expected 5132 images, found {len(df)}')
if not df['read_ok'].astype(bool).all(): raise ValueError('Corrupt/unreadable images exist. Resolve before splitting.')
if df['sha256'].isna().any(): raise ValueError('Missing SHA-256 values.')

cross = df.groupby('sha256')['label'].nunique()
cross = cross[cross > 1]
if len(cross): raise ValueError(f'Cross-label duplicate groups found: {len(cross)}')

counts = df['label'].value_counts().reindex(['Normal','Pneumonia'])
display(counts.to_frame('images'))
print('Integrity gates: PASS')


## 2. Build duplicate-aware groups

Mỗi SHA-256 là một group. Ảnh unique có group size = 1; exact duplicates có group size > 1.

In [ ]:
groups = (df.groupby(['sha256','label'], as_index=False)
            .agg(group_size=('filename','size')))
print('Unique SHA groups:', len(groups))
print('Duplicate groups:', int((groups['group_size'] > 1).sum()))
display(groups['group_size'].value_counts().sort_index().to_frame('groups'))


## 3. Assign locked 15% holdout by label, without splitting duplicate groups

In [ ]:
rng = np.random.default_rng(SEED)
holdout_hashes = set()
holdout_summary = {}

for label in ['Normal','Pneumonia']:
    g = groups[groups['label'] == label].copy()
    g = g.iloc[rng.permutation(len(g))].reset_index(drop=True)
    target = int(round((df['label'] == label).sum() * HOLDOUT_FRACTION))
    selected = []
    running = 0
    for _, row in g.iterrows():
        if running >= target: break
        selected.append(row['sha256'])
        running += int(row['group_size'])
    holdout_hashes.update(selected)
    holdout_summary[label] = {'target_images': target, 'actual_images': running, 'groups': len(selected)}

df['split'] = np.where(df['sha256'].isin(holdout_hashes), 'holdout', 'dev')
display(pd.crosstab(df['split'], df['label']))
print(json.dumps(holdout_summary, indent=2))


## 4. Assign 5 duplicate-aware CV folds inside dev set

Trong mỗi fold `k`: validation = dev rows có `cv_fold == k`; training = các dev rows còn lại.

In [ ]:
df['cv_fold'] = -1
dev_groups = groups[~groups['sha256'].isin(holdout_hashes)].copy()
fold_image_counts = {label:[0]*N_FOLDS for label in ['Normal','Pneumonia']}

for label in ['Normal','Pneumonia']:
    g = dev_groups[dev_groups['label'] == label].copy()
    # Randomize first, then place larger duplicate groups early to balance fold image counts.
    g = g.iloc[rng.permutation(len(g))].copy()
    g = g.sort_values('group_size', ascending=False, kind='stable')
    assignment = {}
    for _, row in g.iterrows():
        min_count = min(fold_image_counts[label])
        candidate_folds = [i for i,c in enumerate(fold_image_counts[label]) if c == min_count]
        fold = int(rng.choice(candidate_folds))
        assignment[row['sha256']] = fold
        fold_image_counts[label][fold] += int(row['group_size'])
    mask = (df['split'] == 'dev') & (df['label'] == label)
    df.loc[mask, 'cv_fold'] = df.loc[mask, 'sha256'].map(assignment).astype(int)

if (df.loc[df['split']=='dev','cv_fold'] < 0).any(): raise RuntimeError('Some dev rows were not assigned to a CV fold.')
display(pd.crosstab(df.loc[df['split']=='dev','cv_fold'], df.loc[df['split']=='dev','label']))
print('Fold image counts:', fold_image_counts)


## 5. Leakage checks

In [ ]:
# A SHA group must map to exactly one split and, for dev, exactly one fold.
split_nunique = df.groupby('sha256')['split'].nunique()
assert split_nunique.max() == 1

dev = df[df['split']=='dev']
fold_nunique = dev.groupby('sha256')['cv_fold'].nunique()
assert fold_nunique.max() == 1

holdout_hash_set = set(df.loc[df['split']=='holdout','sha256'])
dev_hash_set = set(df.loc[df['split']=='dev','sha256'])
assert holdout_hash_set.isdisjoint(dev_hash_set)

print('PASS | no exact-duplicate leakage between dev and holdout')
print('PASS | no exact-duplicate leakage across CV folds')


## 6. Export manifests and protocol summary

In [ ]:
manifest_cols = ['filename','path','label','sha256','split','cv_fold','width','height','aspect_ratio','mode','channels','brightness','contrast']
available_cols = [c for c in manifest_cols if c in df.columns]
manifest = df[available_cols].sort_values(['split','cv_fold','label','filename']).reset_index(drop=True)
manifest_path = OUTPUT_DIR/'split_manifest.csv'
manifest.to_csv(manifest_path,index=False)

fold_rows = []
for fold in range(N_FOLDS):
    fold_rows.append({
        'fold': fold,
        'train_images': int(((df['split']=='dev') & (df['cv_fold']!=fold)).sum()),
        'val_images': int(((df['split']=='dev') & (df['cv_fold']==fold)).sum()),
        'val_normal': int(((df['split']=='dev') & (df['cv_fold']==fold) & (df['label']=='Normal')).sum()),
        'val_pneumonia': int(((df['split']=='dev') & (df['cv_fold']==fold) & (df['label']=='Pneumonia')).sum()),
    })
fold_table = pd.DataFrame(fold_rows)
fold_table.to_csv(OUTPUT_DIR/'cv_fold_summary.csv',index=False)
display(fold_table)

protocol = {
  'seed': SEED,
  'total_images': int(len(df)),
  'holdout_fraction_requested': HOLDOUT_FRACTION,
  'holdout_images': int((df['split']=='holdout').sum()),
  'dev_images': int((df['split']=='dev').sum()),
  'holdout_by_class': {k:int(v) for k,v in df[df['split']=='holdout']['label'].value_counts().to_dict().items()},
  'dev_by_class': {k:int(v) for k,v in df[df['split']=='dev']['label'].value_counts().to_dict().items()},
  'n_cv_folds': N_FOLDS,
  'duplicate_policy': 'all identical SHA-256 images stay in the same split/fold',
  'holdout_policy': 'locked; do not use for architecture/hyperparameter/threshold selection'
}
with open(OUTPUT_DIR/'split_protocol.json','w',encoding='utf-8') as f: json.dump(protocol,f,indent=2,ensure_ascii=False)
print(json.dumps(protocol,indent=2,ensure_ascii=False))
print('Saved:', manifest_path)


## 7. How to use this protocol

1. Không nhìn holdout trong lúc chọn model/hyperparameter/threshold.
2. Với mỗi experiment, chạy 5 folds trên `dev` và báo cáo mean ± std của competition score.
3. Chỉ sau khi chốt pipeline, đánh giá **một lần** trên internal holdout.
4. Nếu thay đổi pipeline sau khi xem holdout, holdout không còn là test nội bộ hợp lệ nữa.
